# Exploratory Data Analysis

| label | class |
| --- | --- |
| 0 | World |
| 1 | Sports |
| 2 | Business |
| 3 | Sci/Tech |


In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", 200)

DATA_DIR = Path("../data/downloads")
CLASS_NAMES = ["World", "Sports", "Business", "Sci/Tech"]

In [2]:
train = pd.read_parquet(DATA_DIR / "train.parquet")
val = pd.read_parquet(DATA_DIR / "validation.parquet")

# Parquet stores `label` as an int; add a readable class column for the analysis.
for df in (train, val):
    df["class"] = df["label"].map(dict(enumerate(CLASS_NAMES)))

print(f"train: {len(train):,} rows | validation: {len(val):,} rows")
train.head(3)

train: 5,000 rows | validation: 1,000 rows


,text,label,class
0,"Bangladesh paralysed by strikes Opposition activists have brought many towns and cities in Bangladesh to a halt, the day after 18 people died in explosions at a political rally.",0,World
1,Desiring Stability Redskins coach Joe Gibbs expects few major personnel changes in the offseason and wants to instill a culture of stability in Washington.,1,Sports
2,"Will Putin #39;s Power Play Make Russia Safer? Outwardly, Russia has not changed since the barrage of terrorist attacks that culminated in the school massacre in Beslan on Sept.",0,World


## Schema & types

Both splits share the same two columns produced by the fetcher (`text`, `label`);
`class` is the readable label we just added.

In [3]:
print("Columns:", list(train.columns))
train.dtypes

Columns: ['text', 'label', 'class']


text       str
label    int64
class      str
dtype: object

## Class distribution

AG News is balanced by design. We check that our random subset kept that balance.
If it did, plain accuracy is a meaningful metric (no single class dominates),
and we will still report macro-F1 as a robustness check.

In [4]:
dist = pd.DataFrame({
    "train": train["class"].value_counts(),
    "train_%": (train["class"].value_counts(normalize=True) * 100).round(1),
    "validation": val["class"].value_counts(),
    "validation_%": (val["class"].value_counts(normalize=True) * 100).round(1),
})
dist.loc[CLASS_NAMES]

,train,train_%,validation,validation_%
class,,,,
World,1253,25.1,266,26.6
Sports,1273,25.5,246,24.6
Business,1150,23.0,246,24.6
Sci/Tech,1324,26.5,242,24.2


## Text length

How long are the snippets? This directly informs the `max_length` used when
tokenizing for DistilBERT. Word count is a rough proxy for token count - the
real tokenizer splits into sub-words, so token counts run somewhat higher.

In [5]:
train["n_chars"] = train["text"].str.len()
train["n_words"] = train["text"].str.split().str.len()

train[["n_chars", "n_words"]].describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(1)

,n_chars,n_words
count,5000.0,5000.0
mean,235.6,37.7
std,65.0,9.8
min,100.0,11.0
50%,232.0,37.0
90%,300.0,48.0
95%,341.0,52.0
99%,456.0,68.0
max,970.0,144.0


Average length per class, in case some topics are systematically longer.

In [6]:
train.groupby("class")[["n_chars", "n_words"]].mean().round(1).loc[CLASS_NAMES]

,n_chars,n_words
class,,
World,242.1,38.8
Sports,223.6,37.6
Business,242.4,37.7
Sci/Tech,234.9,36.9


## Example rows per class

A couple of real snippets per topic, to get a feel for the writing style.

In [7]:
for name in CLASS_NAMES:
    print(f"\n=== {name} ===")
    for text in train.loc[train["class"] == name, "text"].head(2):
        print(" -", text)


=== World ===
 - Bangladesh paralysed by strikes Opposition activists have brought many towns and cities in Bangladesh to a halt, the day after 18 people died in explosions at a political rally.
 - Will Putin #39;s Power Play Make Russia Safer? Outwardly, Russia has not changed since the barrage of terrorist attacks that culminated in the school massacre in Beslan on Sept.

=== Sports ===
 - Desiring Stability Redskins coach Joe Gibbs expects few major personnel changes in the offseason and wants to instill a culture of stability in Washington.
 - Mutombo says he #39;s being traded to Rockets; will back up, mentor &lt;b&gt;...&lt;/b&gt; Dikembe Mutombo, 38, has agreed to a sign-and-trade deal that will send him from the Chicago Bulls to Houston in exchange for Eric Piatkowski, Adrian Griffin and Mike Wilks, the Houston Chronicle reports.

=== Business ===
 - Economy builds steam in KC Fed district The economy continued to strengthen in September and early October in the Great Plains a

## Data quality

Quick sanity checks: missing values and exact-duplicate texts.

In [8]:
print("Missing values per column:")
print(train[["text", "label"]].isna().sum())
print(f"\nExact-duplicate texts in train: {train['text'].duplicated().sum()}")

Missing values per column:
text     0
label    0
dtype: int64

Exact-duplicate texts in train: 0
